In [1]:
import time
import re
import pytz
import pandas as pd
from datetime import datetime
import threading

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import selenium.common.exceptions as se

amazon_base = "https://www.amazon.com.tr"
amazon_login_page = "https://www.amazon.com.tr/ap/signin?openid.pape.max_auth_age=0&openid.return_to=https%3A%2F%2Fwww.amazon.com.tr%2F%3Fref_%3Dnav_ya_signin&openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.assoc_handle=trflex&openid.mode=checkid_setup&openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0&openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0"

TR_MONTHS = {
    "Ocak": 1, "Şubat": 2, "Mart": 3, "Nisan": 4, "Mayıs": 5, "Haziran": 6,
    "Temmuz": 7, "Ağustos": 8, "Eylül": 9, "Ekim": 10, "Kasım": 11, "Aralık": 12
}

# Saatte 100 ürün = ürün başına minimum 36 saniye
PRODUCTS_PER_HOUR = 50
MIN_SECONDS_PER_PRODUCT = 3600 / PRODUCTS_PER_HOUR

def make_driver(detach=True, headless=False):
    chrome_options = webdriver.ChromeOptions()
    if detach:
        chrome_options.add_experimental_option("detach", True)
    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1400,900")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver


In [2]:
class StopController:
    """
    Windows/Jupyter için pratik durdurma kontrolü.
    - Arka planda ENTER bekleyen thread başlatır.
    - Windows terminalde ayrıca tek tuş (q) ile de durdurabilir.
    """
    def __init__(self):
        self.stop_requested = False
        self.stop_reason = None
        self._input_thread = None
        self._input_started = False

    def start(self):
        if self._input_started:
            return
        self._input_started = True

        def _listen():
            try:
                user_val = input("Kazımayı durdurmak için 'q' yazıp ENTER'a bas.\n")
                if str(user_val).strip().lower() == "q":
                    self.request_stop("Kullanıcı 'q' + ENTER ile durdurdu.")
            except Exception:
                pass

        self._input_thread = threading.Thread(target=_listen, daemon=True)
        self._input_thread.start()

    def request_stop(self, reason="Stop requested"):
        self.stop_requested = True
        self.stop_reason = reason

    def check_stop(self):
        if self.stop_requested:
            return True

        # Windows terminal için tek tuş desteği
        try:
            import msvcrt
            if msvcrt.kbhit():
                ch = msvcrt.getwch()
                if str(ch).lower() == "q":
                    self.request_stop("Kullanıcı 'q' tuşuna bastı.")
                    return True
        except Exception:
            pass

        return self.stop_requested


def save_progress(all_dfs, output_csv_path="amazon_tr_reviews_local_only.csv", drop_review_id=True):
    if all_dfs:
        out = pd.concat(all_dfs, ignore_index=True)
    else:
        out = pd.DataFrame(columns=[
            "review_id", "asin", "title", "rating", "body", "review_date",
            "product_title", "product_description", "product_url",
            "source", "review_summary", "collected_at"
        ])

    if drop_review_id and "review_id" in out.columns:
        out = out.drop(columns=["review_id"])

    out.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    return out


def maybe_wait_product_rate_limit(start_ts, min_seconds_per_product=MIN_SECONDS_PER_PRODUCT, stop_controller=None):
    elapsed = time.time() - start_ts
    remaining = max(0, min_seconds_per_product - elapsed)
    if remaining <= 0:
        return

    print(f"  -> rate limit için {remaining:.1f} sn bekleniyor (hedef: saatte 100 ürün)")
    slept = 0.0
    while slept < remaining:
        if stop_controller and stop_controller.check_stop():
            break
        step = min(0.5, remaining - slept)
        time.sleep(step)
        slept += step


In [3]:
def build_review_url(asin: str, page: int) -> str:
    ref = "cm_cr_arp_d_viewopt_sr" if page <= 1 else f"cm_cr_arp_d_paging_btm_next_{page}"
    return (
        f"{amazon_base}/product-reviews/{asin}/ref={ref}"
        f"?ie=UTF8&reviewerType=all_reviews&sortBy=recent&pageNumber={page}"
    )

def build_product_url(asin: str) -> str:
    return f"{amazon_base}/dp/{asin}"

In [4]:
def wait_reviews_stable(driver, timeout=20, stable_rounds=3, poll=0.25):
    end = time.time() + timeout
    last = -1
    stable = 0
    while time.time() < end:
        cards = driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')
        n = len(cards)
        if n == last and n > 0:
            stable += 1
            if stable >= stable_rounds:
                return n
        else:
            stable = 0
            last = n
        time.sleep(poll)
    return len(driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]'))


In [5]:
def safe_text(el) -> str:
    try:
        return el.text.strip()
    except Exception:
        return ""

def parse_amazon_tr_rating(text: str):
    if not text:
        return None
    m = re.search(r"üzerinden\s+(\d+(?:[.,]\d+)?)", text)
    if not m:
        m = re.search(r"(\d+(?:[.,]\d+)?)", text)
    if not m:
        return None
    return float(m.group(1).replace(",", "."))


def parse_amazon_tr_date_to_iso(date_text: str):
    if not date_text:
        return None

    m = re.search(r"(\d{1,2})\s+([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+(\d{4})", date_text)
    if not m:
        return None

    day = int(m.group(1))
    mon_name = m.group(2).strip()
    year = int(m.group(3))
    mon = TR_MONTHS.get(mon_name)
    if not mon:
        return None
    return datetime(year, mon, day).date().isoformat()

In [6]:
def get_product_meta(driver, asin: str, wait_timeout: int = 15) -> dict:
    wait = WebDriverWait(driver, wait_timeout)

    product_url = build_product_url(asin)
    driver.get(product_url)

    product_title = None
    try:
        title_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#productTitle")))
        product_title = title_el.text.strip()
    except Exception:
        product_title = None

    product_description = None
    desc_candidates = [
        "#feature-bullets ul li span.a-list-item",
        "ul.a-unordered-list.a-vertical li span.a-list-item",
        "#productDescription",
    ]

    bullets = []
    for sel in desc_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            texts = [e.text.strip() for e in els if e.text and e.text.strip()]
            if texts:
                if sel == "#productDescription" and not bullets:
                    product_description = " ".join(texts).strip()
                    break
                else:
                    bullets.extend(texts)
        except Exception:
            continue

    if bullets:
        product_description = " | ".join(dict.fromkeys(bullets))

    return {
        "product_title": product_title,
        "product_description": product_description,
        "product_url": product_url
    }

In [7]:
def find_local_and_global_cards(driver):
    container = driver.find_element(By.CSS_SELECTOR, "#cm_cr-review_list")

    cards = container.find_elements(By.CSS_SELECTOR, 'li[data-hook="review"], div[data-hook="review"]')

    local_headers = container.find_elements(By.CSS_SELECTOR, 'h3[data-hook="arp-local-reviews-header"]')
    local_text = local_headers[0].text.strip() if local_headers else ""

    is_local_section = ("Ülke:" in local_text) or ("Türkiye" in local_text)

    if is_local_section:
        return cards, []
    else:
        return [], cards

In [8]:
def extract_rating_from_card_text(card) -> float | None:
    try:
        txt = card.get_attribute("innerText") or card.get_attribute("textContent") or ""
        txt = txt.replace("\u00a0", " ").strip()  # nbsp temizle
    except Exception:
        txt = (card.text or "").strip()

    if not txt:
        return None

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*yıldız\s*üzerinden\s*(\d+(?:[.,]\d+)?)",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(2).replace(",", "."))

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*üzerinden\s*(\d+(?:[.,]\d+)?)",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(2).replace(",", "."))

    m = re.search(
        r"(\d+(?:[.,]\d+)?)\s*yıldız",
        txt,
        flags=re.IGNORECASE
    )
    if m:
        return float(m.group(1).replace(",", "."))

    return None

In [9]:
def parse_review_card_to_row(card, asin: str, current_reviews_url: str, product_meta: dict = None):
    # Title
    title = None
    try:
        title_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-title"]')
        title = safe_text(title_el)
    except Exception:
        pass

    # Rating -> float
    rating_val = extract_rating_from_card_text(card)

    # Body
    body = None
    try:
        body_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-body"]')
        body = safe_text(body_el).replace("\n", " ").strip()
    except Exception:
        pass

    # Date -> ISO
    review_date_iso = None
    try:
        date_el = card.find_element(By.CSS_SELECTOR, 'span[data-hook="review-date"]')
        review_date_iso = parse_amazon_tr_date_to_iso(safe_text(date_el))
    except Exception:
        pass

    review_id = None
    try:
        review_id = card.get_attribute("id")  # R32D2E5N2T6S02
    except Exception:
        pass

    turkey_tz = pytz.timezone("Europe/Istanbul")
    collected_at = datetime.now(turkey_tz).strftime("%Y-%m-%d %H:%M")

    return {
            "review_id": review_id,
            "asin": asin,
            "title": title,
            "rating": rating_val,
            "body": body,
            "review_date": review_date_iso,
    
            # product meta
            "product_title": (product_meta or {}).get("product_title"),
            "product_description": (product_meta or {}).get("product_description"),
    
            "product_url": build_product_url(asin),
            "source": "amazon_tr",
            "review_summary": None,
            "collected_at": collected_at,
    }

In [10]:
def scrape_most_recent_local_reviews_df(
    driver,
    asin: str,
    target_n: int = 80,
    max_pages: int = 80,
    wait_timeout: int = 15,
    stop_controller=None
):
    wait = WebDriverWait(driver, wait_timeout)
    seen = set()
    rows = []

    # --- product meta (visit product page once) ---
    try:
        product_meta = get_product_meta(driver, asin, wait_timeout=wait_timeout)
    except Exception:
        product_meta = {"product_title": None, "product_description": None}

    if stop_controller and stop_controller.check_stop():
        return pd.DataFrame(rows)

    driver.get(build_review_url(asin, 1))

    last_url = None
    stagnant_rounds = 0

    for page in range(1, max_pages + 1):
        if stop_controller and stop_controller.check_stop():
            print(f"  -> stop algılandı, {asin} için kısmi veriyle çıkılıyor (sayfa {page})")
            break

        # --- wait for reviews to load ---
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#cm_cr-review_list")))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')))
            wait_reviews_stable(driver, timeout=25, stable_rounds=4)
        except se.TimeoutException:
            break

        # --- get review cards (local vs global) ---
        local_cards, global_cards = find_local_and_global_cards(driver)

        # stop if global started or locals ended
        if len(global_cards) > 0 or len(local_cards) == 0:
            break

        page_had_new_reviews = False

        # --- parse cards ---
        for i, card in enumerate(local_cards):
            if stop_controller and stop_controller.check_stop():
                print(f"  -> stop algılandı, {asin} için kısmi veriyle çıkılıyor (kart {i+1})")
                break

            try:
                rid = card.get_attribute("id")
            except Exception:
                rid = None

            if rid and rid in seen:
                continue

            try:
                row = parse_review_card_to_row(card, asin, driver.current_url, product_meta=product_meta)
            except se.StaleElementReferenceException:
                cards2 = driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list li[data-hook="review"], #cm_cr-review_list div[data-hook="review"]')
                if i >= len(cards2):
                    continue
                row = parse_review_card_to_row(cards2[i], asin, driver.current_url, product_meta=product_meta)
                rid = row.get("review_id")

            if rid:
                seen.add(rid)
            rows.append(row)
            page_had_new_reviews = True

            if len(seen) >= target_n:
                df = pd.DataFrame(rows)
                if "review_id" in df.columns:
                    df = df.drop_duplicates(subset=["review_id"])
                else:
                    df = df.drop_duplicates()
                return df.head(target_n)

        if stop_controller and stop_controller.check_stop():
            break

        if page_had_new_reviews:
            stagnant_rounds = 0
        else:
            stagnant_rounds += 1
            if stagnant_rounds >= 2:
                print(f"  -> yeni review gelmedi, çıkılıyor: {asin}")
                break

        time.sleep(0.6)

        current_url_before_click = driver.current_url
        card_count_before_click = len(local_cards)
        navigation_used = None

        # 1) Normal next page mantığı
        try:
            disabled_next = driver.find_elements(By.CSS_SELECTOR, "li.a-disabled.a-last")
            if not disabled_next:
                next_candidates = driver.find_elements(By.CSS_SELECTOR, "li.a-last a")
                if next_candidates:
                    next_a = next_candidates[0]
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_a)
                    try:
                        next_a.click()
                    except Exception:
                        driver.execute_script("arguments[0].click();", next_a)
                    navigation_used = "next"
        except Exception:
            navigation_used = None

        # 2) Bazı ürünlerde klasik next yok, append eden '10 tane daha yorum göster' var
        if navigation_used is None:
            try:
                show_more_candidates = driver.find_elements(By.CSS_SELECTOR, 'a[data-hook="show-more-button"]')
                show_more_candidates = [el for el in show_more_candidates if el.is_displayed()]
                if show_more_candidates:
                    show_more_a = show_more_candidates[0]
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", show_more_a)
                    try:
                        show_more_a.click()
                    except Exception:
                        driver.execute_script("arguments[0].click();", show_more_a)
                    navigation_used = "show_more"
            except Exception:
                navigation_used = None

        if navigation_used is None:
            break

        try:
            if navigation_used == "next":
                wait.until(lambda d: d.current_url != current_url_before_click)
                last_url = driver.current_url
            else:
                wait.until(
                    lambda d: len(d.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]')) > card_count_before_click
                )
                # show-more'da URL aynı kalabilir; bu normal
                if driver.current_url == current_url_before_click:
                    last_url = current_url_before_click
                else:
                    last_url = driver.current_url
        except se.TimeoutException:
            # fallback: bazen URL değişmez ama içerik yüklenmiş olabilir
            current_count = len(driver.find_elements(By.CSS_SELECTOR, '#cm_cr-review_list [data-hook="review"]'))
            if navigation_used == "show_more" and current_count > card_count_before_click:
                pass
            else:
                break

    df = pd.DataFrame(rows[:target_n])

    if "review_id" in df.columns:
        df = df.drop_duplicates(subset=["review_id"])
    else:
        df = df.drop_duplicates()

    return df



In [11]:
def scrape_many_asins(
    driver,
    asins,
    target_n=80,
    max_pages=80,
    output_csv_path="amazon_tr_reviews_local_only.csv",
    min_seconds_per_product=MIN_SECONDS_PER_PRODUCT
):
    all_dfs = []
    stop_controller = StopController()
    stop_controller.start()

    last_completed_asin = None
    stopped_on_asin = None

    for idx, asin in enumerate(asins, 1):
        if stop_controller.check_stop():
            stopped_on_asin = asin
            print("\nDurdurma isteği alındı. Yeni ürüne başlamadan önce çıkılıyor.")
            break

        product_start_ts = time.time()

        try:
            print(f"[{idx}/{len(asins)}] Scraping ASIN: {asin}")
            df = scrape_most_recent_local_reviews_df(
                driver,
                asin=asin,
                target_n=target_n,
                max_pages=max_pages,
                stop_controller=stop_controller
            )
            print(f"  -> got {len(df)} reviews")
            all_dfs.append(df)
            last_completed_asin = asin

            # her ürün sonrası güvenli autosave
            save_progress(all_dfs, output_csv_path=output_csv_path, drop_review_id=True)
            print(f"  -> ara kayıt alındı: {output_csv_path}")

            if stop_controller.check_stop():
                stopped_on_asin = asin
                print(f"\nKazıma kullanıcı isteğiyle durduruldu. Durulan ürün: {asin}")
                break

            maybe_wait_product_rate_limit(
                product_start_ts,
                min_seconds_per_product=min_seconds_per_product,
                stop_controller=stop_controller
            )

            if stop_controller.check_stop():
                stopped_on_asin = asin
                print(f"\nBekleme sırasında durduruldu. Son işlenen ürün: {asin}")
                break

        except Exception as e:
            print(f"  !! FAILED ASIN {asin}: {type(e).__name__}: {e}")

            # hata olsa bile o ana kadarki veriyi kaydet
            save_progress(all_dfs, output_csv_path=output_csv_path, drop_review_id=True)
            continue

    final_df = save_progress(all_dfs, output_csv_path=output_csv_path, drop_review_id=True)

    if stopped_on_asin:
        print(f"\nKazıma durdu.")
        print(f"Durulan ürün ASIN: {stopped_on_asin}")
        print(f"Şu ana kadar kaydedilen satır sayısı: {len(final_df)}")
        print(f"CSV kaydedildi: {output_csv_path}")
    else:
        print(f"\nKazıma tamamlandı. Son tamamlanan ürün: {last_completed_asin}")
        print(f"Toplam satır sayısı: {len(final_df)}")
        print(f"CSV kaydedildi: {output_csv_path}")

    return final_df


In [12]:
driver = make_driver(detach=True, headless=False)

# Login (manuel)
driver.get(amazon_login_page)

In [13]:
asins = [
# Elektrikli Süpürge
"B08S787DSS", "B08P7WBBXP", "B0B6GFY5LB", "B08D6CR4N9", "B0DCC2YQP7",
"B0DSKXG7WB", "B0DCGD7Y8K", "B0DCGD7Y8K", "B0B829K872", "B07YNZWR56",

# Filtre Kahve Makinesi
"B00R04CAH0", "B01534GMCM", "B0BX9MGW6G", "B00R04CAWA", "B09WYWPY14",
"B095SRNR3H", "B01B0YQV66", "B084ZPHMS1", "B00MB6J36I", "B0741Z3WJC",

# Diş Macunu
"B07YQFT2ZV", "B004PCEQZE", "B07BB54JZ4", "B08R9CWLY1", "B07G11C7YW",
"B0BTT2SKDT", "B09SB9P5QZ", "B0DKFZ5X2V", "B0D4F6523X", "B073T33K69",

# Diş Fırçası
"B0DXPNVX74", "B089F3W17W", "B0B515Z52M", "B0B5192LBN", "B0DZ2CF8M1",
"B08GHGMD4N", "B0B51CWCN7", "B0BJMS84K6", "B086DV91S3", "B018907QLI",

# Maskara
"B0BWRTJLZ1", "B0CJ5GW8RY", "B0092QK8N2", "B0CJ5FPMN3", "B00PFCSXD4",
"B07L7Q962B", "B07FDHR2QM", "B0CJ5G8LKV", "B00C7D24IE", "B008OG7V2M",

]

In [13]:
import json

with open("03142026Amazon.txt", "r", encoding="utf-8") as f:
    links = json.load(f)

print(type(links))  # list
asins = [u.split("/dp/")[1][:10] for u in links]

<class 'list'>


In [14]:
index = asins.index("B00HWR2TKI")
index

126

In [15]:
asins = asins[index:]

In [16]:
print(len(asins))

3432


In [17]:
OUTPUT_CSV_PATH = "03142026 126dan.csv"

df_all = scrape_many_asins(
    driver,
    asins,
    target_n=80,
    max_pages=80,
    output_csv_path=OUTPUT_CSV_PATH,
    min_seconds_per_product=MIN_SECONDS_PER_PRODUCT
)
df_all.head()

[1/3432] Scraping ASIN: B000CSVK6I
  -> got 80 reviews
  -> ara kayıt alındı: 03142026 126dan.csv
  -> rate limit için 43.2 sn bekleniyor (hedef: saatte 100 ürün)
[2/3432] Scraping ASIN: B000CZ0RLU
  -> got 80 reviews
  -> ara kayıt alındı: 03142026 126dan.csv
  -> rate limit için 47.2 sn bekleniyor (hedef: saatte 100 ürün)
[3/3432] Scraping ASIN: B000EBN174
  -> got 80 reviews
  -> ara kayıt alındı: 03142026 126dan.csv
  -> rate limit için 47.6 sn bekleniyor (hedef: saatte 100 ürün)
[4/3432] Scraping ASIN: B000EEHJFG
  -> got 80 reviews
  -> ara kayıt alındı: 03142026 126dan.csv
  -> rate limit için 47.4 sn bekleniyor (hedef: saatte 100 ürün)
[5/3432] Scraping ASIN: B000EEJ8D2
  -> got 80 reviews
  -> ara kayıt alındı: 03142026 126dan.csv
  -> rate limit için 47.6 sn bekleniyor (hedef: saatte 100 ürün)
[6/3432] Scraping ASIN: B000EEL6GY
  -> got 80 reviews
  -> ara kayıt alındı: 03142026 126dan.csv
  -> rate limit için 47.0 sn bekleniyor (hedef: saatte 100 ürün)
[7/3432] Scraping ASIN

Kazımayı durdurmak için 'q' yazıp ENTER'a bas.
 q


  -> ara kayıt alındı: 03142026 126dan.csv

Kazıma kullanıcı isteğiyle durduruldu. Durulan ürün: B00HWR2TKI

Kazıma durdu.
Durulan ürün ASIN: B00HWR2TKI
Şu ana kadar kaydedilen satır sayısı: 32902
CSV kaydedildi: 03142026 126dan.csv


,asin,title,rating,body,review_date,product_title,product_description,product_url,source,review_summary,collected_at
0,B000CSVK6I,My husband loved it,5.0,"I am not a hunter, and I have no idea how to u...",2026-02-16,"DUCK COMMANDER Klasik Serisi Ördek Çağrısı, Ör...",Dayanıklı Polikarbonat Yapı: Bu ördek düdüğü s...,https://www.amazon.com.tr/dp/B000CSVK6I,amazon_tr,None,2026-04-05 13:24
1,B000CSVK6I,Easy just blow in to it,5.0,Just fantastic easy use and functional is grea...,2025-12-21,"DUCK COMMANDER Klasik Serisi Ördek Çağrısı, Ör...",Dayanıklı Polikarbonat Yapı: Bu ördek düdüğü s...,https://www.amazon.com.tr/dp/B000CSVK6I,amazon_tr,None,2026-04-05 13:24
2,B000CSVK6I,Loud- not for quiet applications.,5.0,Great quality.,2025-11-24,"DUCK COMMANDER Klasik Serisi Ördek Çağrısı, Ör...",Dayanıklı Polikarbonat Yapı: Bu ördek düdüğü s...,https://www.amazon.com.tr/dp/B000CSVK6I,amazon_tr,None,2026-04-05 13:24
3,B000CSVK6I,Husband free to a good home,5.0,My husband bought this for duck hunting. It so...,2025-10-31,"DUCK COMMANDER Klasik Serisi Ördek Çağrısı, Ör...",Dayanıklı Polikarbonat Yapı: Bu ördek düdüğü s...,https://www.amazon.com.tr/dp/B000CSVK6I,amazon_tr,None,2026-04-05 13:24
4,B000CSVK6I,I have been using this call for years now,5.0,"My favorite call , the one call I always grab",2025-10-06,"DUCK COMMANDER Klasik Serisi Ördek Çağrısı, Ör...",Dayanıklı Polikarbonat Yapı: Bu ördek düdüğü s...,https://www.amazon.com.tr/dp/B000CSVK6I,amazon_tr,None,2026-04-05 13:24


In [18]:
df_all.tail()

,asin,title,rating,body,review_date,product_title,product_description,product_url,source,review_summary,collected_at
28113,B09TBD4MCV,En iyisi,5.0,Duracell en iyisi,2023-12-04,Duracell Optimum AAA İnce Kalem Pil (6’lı Pake...,Cihazlarınızı geliştirin: Power Boost aktifler...,https://www.amazon.com.tr/dp/B09TBD4MCV,amazon_tr,None,2026-03-20 18:59
28114,B09TBD4MCV,Duracell Kalitesi,5.0,"Prime indirinden aldim, elime cok hizli ulaşdi...",2023-11-26,Duracell Optimum AAA İnce Kalem Pil (6’lı Pake...,Cihazlarınızı geliştirin: Power Boost aktifler...,https://www.amazon.com.tr/dp/B09TBD4MCV,amazon_tr,None,2026-03-20 18:59
28115,B09TBD4MCV,sağlam kalite,5.0,beğendik,2023-11-22,Duracell Optimum AAA İnce Kalem Pil (6’lı Pake...,Cihazlarınızı geliştirin: Power Boost aktifler...,https://www.amazon.com.tr/dp/B09TBD4MCV,amazon_tr,None,2026-03-20 18:59
28116,B09TBD4MCV,Teşekkürler Amazon,5.0,Duracell gerçekten bu işin uzmanı. Şirket için...,2023-11-17,Duracell Optimum AAA İnce Kalem Pil (6’lı Pake...,Cihazlarınızı geliştirin: Power Boost aktifler...,https://www.amazon.com.tr/dp/B09TBD4MCV,amazon_tr,None,2026-03-20 18:59
28117,B09TBD4MCV,M Ü K E M M E L ***,5.0,Sürekli kullandığım ürün gerçek uzun ömür :),2023-10-06,Duracell Optimum AAA İnce Kalem Pil (6’lı Pake...,Cihazlarınızı geliştirin: Power Boost aktifler...,https://www.amazon.com.tr/dp/B09TBD4MCV,amazon_tr,None,2026-03-20 18:59


In [ ]:
# [113/352] Scraping ASIN: B086CSN1D9
#   -> got 0 reviews
# [114/352] Scraping ASIN: B086DQP567
#   -> got 0 reviews
# [115/352] Scraping ASIN: B086QY2DN2
#   -> got 0 reviews
# [116/352] Scraping ASIN: B086ZSZ3MK

In [18]:
df_all.to_csv("03122026 83tensonra.csv", index=False, encoding="utf-8-sig")

#### 03/13/2026 Amazon
**1325**

#### 03/14/2026 Amazon
**3358**

#### 03/12/2026 Amazon ( array.txt )
**353**

#### Bilgisayar

#### Bebek 

#### Bahce

#### Ayakkabi